# 🏨 Atlantic Haven Hotels — Prédiction d'annulation de réservation
### Hackathon ML & Data Science — M1 ISPM

**Objectif :** prédire `reservation_annulee` (0 = maintenue, 1 = annulée), fournir une probabilité et une décision binaire, et maximiser le **F1-score** sur la classe « annulation », tout en respectant l'ordre temporel des données.

---
## 📌 ÉTAPE 1 : EDA (Analyse Exploratoire des Données) & Préparation Initiale

### 1.1 Setup & Configuration de l'environnement

In [ ]:
# --- ÉTAPE 1 : Imports et configuration globale ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve
)

# Graine aléatoire fixée pour la reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

### 1.2 Chargement des données

In [ ]:
# --- ÉTAPE 1 : Chargement des jeux de données ---
TRAIN_PATH = "reservations_train.csv"
TEST_PATH  = "reservations_test.csv"
DICT_PATH  = "data_dictionary.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
data_dict = pd.read_csv(DICT_PATH)

print(f"Forme du jeu de Train : {df_train.shape}")
print(f"Forme du jeu de Test  : {df_test.shape}")
df_train.head()

### 1.3 Inspection du dictionnaire des variables

In [ ]:
# --- ÉTAPE 1 : Affichage du dictionnaire des données ---
data_dict

### 1.4 Nettoyage et harmonisation des types de données

In [ ]:
# --- ÉTAPE 1 : Correction des types de colonnes (Dates & Binaires) ---
def fix_data_types(df):
    df = df.copy()
    # Conversion des colonnes de dates
    df['date_reservation'] = pd.to_datetime(df['date_reservation'])
    df['date_arrivee'] = pd.to_datetime(df['date_arrivee'])

    # Harmonisation de la colonne binaire 'tarif_remboursable'
    if df['tarif_remboursable'].dtype == 'object':
        df['tarif_remboursable'] = df['tarif_remboursable'].astype(str).str.lower().map({
            'oui': 1, 'non': 0, 'true': 1, 'false': 0, '1': 1, '0': 0
        }).fillna(0).astype(int)

    return df

df_train = fix_data_types(df_train)
df_test = fix_data_types(df_test)

print("✅ Dates et types binaires corrigés avec succès !")

### 1.5 Audit des valeurs manquantes

In [ ]:
# --- ÉTAPE 1 : Bilan des valeurs manquantes ---
print("--- VALEURS MANQUANTES (TRAIN) ---")
missing_train = df_train.isnull().sum()
print(missing_train[missing_train > 0])

print("\n--- VALEURS MANQUANTES (TEST) ---")
missing_test = df_test.isnull().sum()
print(missing_test[missing_test > 0])

### 1.6 Imputation métier intelligente

In [ ]:
# --- ÉTAPE 1 : Traitement métier des valeurs manquantes ---
def process_missing_values(train_df, test_df):
    train = train_df.copy()
    test = test_df.copy()

    # 1. agent_id : NaN correspond à une réservation sans agence (Direct)
    train['agent_id'] = train['agent_id'].fillna('Direct')
    test['agent_id'] = test['agent_id'].fillna('Direct')

    train['est_reservation_directe'] = (train['agent_id'] == 'Direct').astype(int)
    test['est_reservation_directe'] = (test['agent_id'] == 'Direct').astype(int)

    # 2. enfants & demandes_speciales : NaN = 0
    train['enfants'] = train['enfants'].fillna(0).astype(int)
    test['enfants'] = test['enfants'].fillna(0).astype(int)

    train['demandes_speciales'] = train['demandes_speciales'].fillna(0).astype(int)
    test['demandes_speciales'] = test['demandes_speciales'].fillna(0).astype(int)

    # 3. marche_origine : Imputation par le mode sur le Train
    mode_marche = train['marche_origine'].mode()[0]
    train['marche_origine'] = train['marche_origine'].fillna(mode_marche)
    test['marche_origine'] = test['marche_origine'].fillna(mode_marche)

    # 4. prix_moyen_nuit_eur : Médiane par catégorie d'hôtel (calculée sur le Train uniquement)
    medians_par_categorie = train.groupby('categorie_hotel')['prix_moyen_nuit_eur'].median()

    train['prix_moyen_nuit_eur'] = train['prix_moyen_nuit_eur'].fillna(
        train['categorie_hotel'].map(medians_par_categorie)
    )
    test['prix_moyen_nuit_eur'] = test['prix_moyen_nuit_eur'].fillna(
        test['categorie_hotel'].map(medians_par_categorie)
    )

    return train, test

# Exécution
df_train_clean, df_test_clean = process_missing_values(df_train, df_test)

### 1.7 Validation du nettoyage

In [ ]:
# --- ÉTAPE 1 : Vérification qu'il ne reste aucun NaN ---
print("Valeurs manquantes restantes dans Train Clean :", df_train_clean.isnull().sum().sum())
print("Valeurs manquantes restantes dans Test Clean  :", df_test_clean.isnull().sum().sum())

### 1.8 Analyse Visuelle de l'EDA & Split Temporel de Validation

In [ ]:
# --- ÉTAPE 1 : Visualisations EDA & Validation Temporelle ---
df_train = df_train_clean.copy()
df_test = df_test_clean.copy()

# Visualisation des facteurs clés d'annulation
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Taux d'annulation par type d'acompte
sns.barplot(data=df_train, x='type_acompte', y='reservation_annulee', ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title("Taux d'annulation selon le type d'acompte")
axes[0, 0].set_ylabel("Proportion d'annulations")

# 2. Taux d'annulation selon le tarif remboursable
sns.barplot(data=df_train, x='tarif_remboursable', y='reservation_annulee', ax=axes[0, 1], palette='Oranges_r')
axes[0, 1].set_title("Taux d'annulation selon le tarif remboursable")

# 3. Distribution du délai de réservation (lead time)
sns.kdeplot(data=df_train, x='delai_reservation_jours', hue='reservation_annulee', common_norm=False, ax=axes[1, 0], palette=['#4f8c48', '#e77a22'])
axes[1, 0].set_title("Distribution du délai de réservation (Jours)")

# 4. Historique d'annulations
sns.boxplot(data=df_train, x='reservation_annulee', y='annulations_passees', ax=axes[1, 1], palette=['#4f8c48', '#e77a22'])
axes[1, 1].set_title("Annulations passées vs Statut de la réservation")

plt.tight_layout()
plt.show()

# --- ÉTAPE 1 : Création du Split Temporel pour Validation ---
df_train_sorted = df_train.sort_values(by='date_reservation').reset_index(drop=True)

# 80% train initial / 20% validation locale (les dates les plus récentes)
val_size = int(len(df_train_sorted) * 0.20)
split_idx = len(df_train_sorted) - val_size

train_split = df_train_sorted.iloc[:split_idx].copy()
val_split = df_train_sorted.iloc[split_idx:].copy()

print("\n✅ --- SPLIT TEMPOREL EFFECTUÉ ---")
print(f"Train Split : {len(train_split)} lignes ({train_split['date_reservation'].min().date()} à {train_split['date_reservation'].max().date()})")
print(f"Val Split   : {len(val_split)} lignes ({val_split['date_reservation'].min().date()} à {val_split['date_reservation'].max().date()})")